### Imports

In [25]:
!pip install pyopengl
!pip install glfw
!pip install pyglm
!pip install numpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [26]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm
import math
import copy
import json
import time


### Contexto geral do projeto


#### Controles
É possível controlar 3 objetos da nossa cena: Peixe, Nuvem e Sol.
- Peixe: Controle de Rotação, pressione a tecla 'a' para rodar o Peixe anti-horário e 'd' para rodar horário.
- Nuvem: Controle de Escala, pressione a tecla 'w' para aumentar a Nuvem e 's' para diminuir.
- Sol: Controle de Translação, pressione a tecla 'seta para esquerda' para o Sol ir para a esquerda, 'seta para direita' para ir para a direita, 'seta para baixo' para ir para baixo, 'seta para cima' para ir para cima, 'z' para trazer mais fora da tela e 'x' para trazer mais dentro da tela.

### Inicializa Janela

In [27]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
window = glfw.create_window(700, 700, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


(python:51258): Gtk-WARNING **: 15:44:09.142: gtk_disable_setlocale() must be called before gtk_init()


### Shaders

In [28]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

# Request a program and shader slots from GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

# Set shaders source
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

# Compile shaders
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")

glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

# Attach shader objects to the program
glAttachShader(program, vertex)
glAttachShader(program, fragment)

# Build program
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')
    
# Make program the default program
glUseProgram(program)

### Modelos

Nessa seção, definimos modelos básicos, que serviram de base para a construção de todos os elementos da nossa cena.

In [29]:
# Definindo modelos

vertices_list = []
objects_vex = []

# Quadrado
quadrado = [
    (-0.2, -0.2, +0.2),
    (+0.2, -0.2, +0.2),
    (-0.2, +0.2, +0.2),
    (+0.2, +0.2, +0.2)
]
objects_vex.append(["quadrado", len(vertices_list), len(quadrado), 0])
for v in quadrado:
    vertices_list.append(v)

# Circulo
num_vertices = 30 # define a "qualidade" do circulo
pi = 3.14
counter = 0
radius = 0.5
circulo = np.zeros(num_vertices, [("position", np.float32, 3)])

angle = 0.0
for counter in range(num_vertices):
    angle += 2*pi/num_vertices 
    x = math.cos(angle)*radius
    y = math.sin(angle)*radius
    circulo[counter] = [x,y, 0.0]

objects_vex.append(["circulo", len(vertices_list), len(circulo), 1])
for v in circulo:
    vertices_list.append(v['position'])

# Triangulo
triangulo = [
                ( 0.0, 0.0, 0.0), # vertice 0
                ( 0.0,+0.5, 0.0), # vertice 1
                (+0.5, 0.0, 0.0), # vertice 2
            ]

objects_vex.append(["triangulo", len(vertices_list), len(triangulo), 0])
for v in triangulo:
    vertices_list.append(v)

# Cubo
cubo = [
    # Face 1 do Cubo (vértices do quadrado)
    (-0.2, -0.2, +0.2),
    (+0.2, -0.2, +0.2),
    (-0.2, +0.2, +0.2),
    (+0.2, +0.2, +0.2),

    # Face 2 do Cubo
    (+0.2, -0.2, +0.2),
    (+0.2, -0.2, -0.2),         
    (+0.2, +0.2, +0.2),
    (+0.2, +0.2, -0.2),
    
    # Face 3 do Cubo
    (+0.2, -0.2, -0.2),
    (-0.2, -0.2, -0.2),            
    (+0.2, +0.2, -0.2),
    (-0.2, +0.2, -0.2),

    # Face 4 do Cubo
    (-0.2, -0.2, -0.2),
    (-0.2, -0.2, +0.2),         
    (-0.2, +0.2, -0.2),
    (-0.2, +0.2, +0.2),

    # Face 5 do Cubo
    (-0.2, -0.2, -0.2),
    (+0.2, -0.2, -0.2),         
    (-0.2, -0.2, +0.2),
    (+0.2, -0.2, +0.2),
    
    # Face 6 do Cubo
    (-0.2, +0.2, +0.2),
    (+0.2, +0.2, +0.2),           
    (-0.2, +0.2, -0.2),
    (+0.2, +0.2, -0.2)
]

objects_vex.append(["cubo", len(vertices_list), len(cubo), 0])
for v in cubo:
    vertices_list.append(v)

# Pirâmide
piramide = [
    # Face 1 da Piramide (triangulo)
    (0.0, 0.5, 0),
    (+0.25, 0.0, -0.25),
    (+0.25, 0.0, +0.25),

    # Face 2 da Piramide (triangulo)
    (0.0, 0.5, 0),
    (+0.25, 0.0, +0.25),
    (-0.25, 0.0, +0.25),

    # Face 3 da Piramide (triangulo)
    (0.0, 0.5, 0),
    (-0.25, 0.0, +0.25),
    (-0.25, 0.0, -0.25),
    
    # Face 4 da Piramide (triangulo)
    (0.0, 0.5, 0),
    (-0.25, 0.0, -0.25),
    (+0.25, 0.0, -0.25),
    
    # Face 5 (base) da Piramide (quadrado)
    (+0.25, 0.0, +0.25),
    (0.25, 0.0, -0.25),
    (-0.25, 0.0, +0.25),
    (-0.25, 0.0, -0.25),
]

objects_vex.append(["piramide", len(vertices_list), len(piramide), 0])
for v in piramide:
    vertices_list.append(v)

# Cilindro
PI = 3.141592
r = 0.1 # raio
H = 0.9
num_sectors = 20 # qtd de sectors (longitude)
num_stacks = 20 # qtd de stacks (latitude)

# grid sectos vs stacks (longitude vs latitude)
sector_step = (PI*2)/num_sectors # variar de 0 até 2π
stack_step = H/num_stacks # variar de 0 até H

# Entrada: angulo de t, altura h, raio r
# Saida: coordenadas no cilindro
def CoordCilindro(t, h, r):
    x = r * math.cos(t)
    y = r * math.sin(t)
    z = h
    return (x,y,z)

# vamos gerar um conjunto de vertices 
# cada poligono eh representado por dois triangulos
cilindro = []
for j in range(0,num_stacks): # para cada stack (latitude)
    
    for i in range(0,num_sectors): # para cada sector (longitude) 
        
        u = i * sector_step # angulo setor
        v = j * stack_step # altura da stack
        
        un = 0 # angulo do proximo sector
        if i+1==num_sectors:
            un = PI*2
        else: un = (i+1)*sector_step
            
        vn = 0 # altura da proxima stack
        if j+1==num_stacks:
            vn = H
        else: vn = (j+1)*stack_step
        
        # verticies do poligono
        p0=CoordCilindro(u, v, r)
        p1=CoordCilindro(u, vn, r)
        p2=CoordCilindro(un, v, r)
        p3=CoordCilindro(un, vn, r)
        
        # triangulo 1 (primeira parte do poligono)
        cilindro.append(p0)
        cilindro.append(p2)
        cilindro.append(p1)
        
        # triangulo 2 (segunda e ultima parte do poligono)
        cilindro.append(p3)
        cilindro.append(p1)
        cilindro.append(p2)
        
        #print(v)
        
        if v == 0:
            cilindro.append(p0)
            cilindro.append(p2)
            cilindro.append(CoordCilindro(0, v, 0))
        if vn == H:
            #faz um triangulo a partir do mesmo angulo u, mas com as alturas em h = vn
            cilindro.append(p1)
            cilindro.append(p3)
            cilindro.append(CoordCilindro(0, vn, 0))

total_vertices = len(cilindro)
cilindro_vertices = np.array(cilindro)

objects_vex.append(["cilindro", len(vertices_list), len(cilindro_vertices), 0])
for v in cilindro_vertices:
    vertices_list.append(v)

# Finalização
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


### Dados

In [30]:
# Request a buffer slot from GPU
buffer_VBO = glGenBuffers(1)
# Make this buffer the default one
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

# Upload data
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

stride = vertices.strides[0]
offset = ctypes.c_void_p(0)

loc = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc)

glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

# Pega loc das cores
loc_color = glGetUniformLocation(program, "color")

### Transformações

In [31]:
def multiplica_matriz(a,b):
    m_a = a.reshape(4,4)
    m_b = b.reshape(4,4)
    m_c = np.dot(m_a,m_b)
    c = m_c.reshape(1,16)
    return c

def multiplica_matriz3(a,b,c):
    return multiplica_matriz(a, multiplica_matriz(b, c))

def mat_rotation_x(angle):
    cos = math.cos(angle)
    sin = math.sin(angle)

    return np.array([   1.0, 0.0,  0.0, 0.0, 
                        0.0, cos, -sin, 0.0, 
                        0.0, sin,  cos, 0.0, 
                        0.0, 0.0,  0.0, 1.0], np.float32)
def mat_rotation_y(angle):
    cos = math.cos(angle)
    sin = math.sin(angle)

    return np.array([    cos, 0.0, sin, 0.0, 
                         0.0, 1.0, 0.0, 0.0, 
                        -sin, 0.0, cos, 0.0, 
                         0.0, 0.0, 0.0, 1.0], np.float32)

def mat_rotation_z(angle):
    cos = math.cos(angle)
    sin = math.sin(angle)

    return np.array([   cos, -sin, 0.0, 0.0, 
                        sin,  cos, 0.0, 0.0, 
                        0.0,  0.0, 1.0, 0.0, 
                        0.0,  0.0, 0.0, 1.0], np.float32)

def mat_scale(sx, sy, sz):
    return np.array([    sx, 0.0, 0.0, 0.0, 
                        0.0,  sy, 0.0, 0.0, 
                        0.0, 0.0,  sz, 0.0, 
                        0.0, 0.0, 0.0, 1.0], np.float32)

def mat_translocate(tx, ty, tz):
    return np.array([   1.0, 0.0, 0.0, tx, 
                        0.0, 1.0, 0.0, ty, 
                        0.0, 0.0, 1.0, tz, 
                        0.0, 0.0, 0.0, 1.0], np.float32)


def calcula_centro_eixos(obj_array, vertices, groups, groupMode):

    if groupMode:
        gid = obj_array[0]['group_id']
        group_des = groups[gid]['deslocamento']
    else:
        group_des = [0.0, 0.0, 0.0]

    m = []
    for obj in obj_array:
        vi = obj['vi']
        vt = obj['vt']

        for u, v, w in vertices['position'][vi:(vi+vt)]:
            u = float(u) + obj['deslocamento'][0] + group_des[0]
            v = float(v) + obj['deslocamento'][1] + group_des[1]
            w = float(w) + obj['deslocamento'][2] + group_des[2]

            if len(m) == 0:
                m.append(u)
                m.append(u)
                m.append(v)
                m.append(v)
                m.append(w)
                m.append(w)

            m[0] = min(m[0], u)
            m[1] = max(m[1], u)
            m[2] = min(m[2], v)
            m[3] = max(m[3], v)
            m[4] = min(m[4], w)
            m[5] = max(m[5], w)
        
    meio = [(m[0]+m[1])/2.0, (m[2]+m[3])/2.0, (m[4]+m[5])/2.0]
    return meio

def to_origin_and_back(scene, obj, mat, vertices, groups, groupMode):
    if groupMode and obj['group_id'] != -1:
        objects = [o for o in scene if o['group_id'] == obj['group_id']]
        centro = calcula_centro_eixos(objects, vertices, groups, groupMode)
    else:
        centro = calcula_centro_eixos([obj], vertices, groups, groupMode)
    
    mat_to_origin = mat_translocate(-centro[0], -centro[1], -centro[2])
    mat_return = mat_translocate(centro[0], centro[1], centro[2])

    return multiplica_matriz3(mat_return, mat, mat_to_origin)


### Controlador de Cena

In [32]:
def load_scene():
    scene = []
    with open("scene.json", "r", encoding="utf-8") as f:
        scene = json.load(f)
    
    return scene

def save_scene(scene):
    with open("scene.json", "w", encoding="utf-8") as f:
        json.dump(scene, f, ensure_ascii=False, indent=4)

def add_to_scene(scene, obj_id, name, color):
    newObj = {
        "name": name,
        "vi": objects_vex[obj_id][1],
        "vt": objects_vex[obj_id][2],
        "deslocamento": [
            0.0,
            0.0,
            0.0
        ],
        "escala": [
            1.0,
            1.0,
            1.0
        ],
        "rotacao": [
            0.0,
            0.0,
            0.0
        ],
        "cor": [
            color[0],
            color[1],
            color[2],
            color[3]
        ],
        "tex_id": -1,
        "obj_id": obj_id,
        "visible": True,
        "polygon": False,
        "scene_id": len(scene),
        "draw_mode": objects_vex[obj_id][3],
        "group_id": -1
    }

    scene.append(newObj)

    return scene




### Eventos

In [33]:
eixo = 1
selected_obj = 0
uniform_scale = True
PolygonMode = False
groupMode = True
restrictMode = True

def key_event(window,key,scancode,action,mods):
    global eixo, selected_obj, scene, uniform_scale, PolygonMode, groups, restrictMode

    if restrictMode:
        # Toggle Polygon Mode
        if key == glfw.KEY_P and action == glfw.PRESS: 
            PolygonMode = not PolygonMode
        
        # Rotação (Peixe)
        if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
            groups[2]['rotacao'][2] += 0.05
        if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
            groups[2]['rotacao'][2] -= 0.05    

        # Escala (Nuvem)
        if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
            groups[1]['escala'][0] += 0.01
            groups[1]['escala'][1] += 0.01
            groups[1]['escala'][2] += 0.01
        if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
            if groups[1]['escala'][0] >= 0:
                groups[1]['escala'][0] -= 0.01
                groups[1]['escala'][1] -= 0.01
                groups[1]['escala'][2] -= 0.01
        
        # Translação (Sol)
        ## X
        if key == glfw.KEY_LEFT and (action == glfw.PRESS or action == glfw.REPEAT):
            groups[0]['deslocamento'][0] -= 0.01
        if key == glfw.KEY_RIGHT and (action == glfw.PRESS or action == glfw.REPEAT):
            groups[0]['deslocamento'][0] += 0.01
        ## Y
        if key == glfw.KEY_DOWN and (action == glfw.PRESS or action == glfw.REPEAT):
            groups[0]['deslocamento'][1] -= 0.01
        if key == glfw.KEY_UP and (action == glfw.PRESS or action == glfw.REPEAT):
            groups[0]['deslocamento'][1] += 0.01
        ## Z
        if key == glfw.KEY_Z and (action == glfw.PRESS or action == glfw.REPEAT):
            groups[0]['deslocamento'][2] -= 0.01
        if key == glfw.KEY_X and (action == glfw.PRESS or action == glfw.REPEAT):
            groups[0]['deslocamento'][2] += 0.01
        

    else:
        # Muda Objeto
        if key == glfw.KEY_Y and action == glfw.PRESS:
            selected_obj = (selected_obj+1) % len(scene)

        # Toggle Polygon Mode
        if key == glfw.KEY_P and action == glfw.PRESS: 
            PolygonMode = not PolygonMode

        # Toggle Visibility
        if key == glfw.KEY_V and action == glfw.PRESS: 
            scene[selected_obj]['visible'] = not scene[selected_obj]['visible']

        # Toggle uniform_scale
        if key == glfw.KEY_C and action == glfw.PRESS:
            uniform_scale = not uniform_scale

        # Troca de Eixo Rotação
        if key == glfw.KEY_Q and action == glfw.PRESS:
            eixo = (eixo+1)%3

        # Rotação
        if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['rotacao'][eixo] += 0.05
        if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['rotacao'][eixo] -= 0.05    

        # Escala
        if uniform_scale:
            if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][0] += 0.01
                scene[selected_obj]['escala'][1] += 0.01
                scene[selected_obj]['escala'][2] += 0.01
            if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][0] -= 0.01
                scene[selected_obj]['escala'][1] -= 0.01
                scene[selected_obj]['escala'][2] -= 0.01
        else:
            if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][eixo] += 0.01
            if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][eixo] -= 0.01

        # Translação
        ## X
        if key == glfw.KEY_LEFT and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['deslocamento'][0] -= 0.01
        if key == glfw.KEY_RIGHT and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['deslocamento'][0] += 0.01
        ## Y
        if key == glfw.KEY_DOWN and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['deslocamento'][1] -= 0.01
        if key == glfw.KEY_UP and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['deslocamento'][1] += 0.01
        ## Z
        if key == glfw.KEY_Z and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['deslocamento'][2] -= 0.01
        if key == glfw.KEY_X and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['deslocamento'][2] += 0.01

glfw.set_key_callback(window,key_event)


### Janela

In [34]:
glfw.show_window(window)

In [35]:
# Groups
groups = []
newGroup = {
            "name": "sun",
            "deslocamento": [
                0.0,
                0.0,
                0.0
            ],
            "escala": [
                1.0,
                1.0,
                1.0
            ],
            "rotacao": [
                0.0,
                0.0,
                0.0
            ],
            "group_id": 0
        }
groups.append(newGroup)

newGroup = {
            "name": "cloud",
            "deslocamento": [
                0.0,
                0.0,
                0.0
            ],
            "escala": [
                1.0,
                1.0,
                1.0
            ],
            "rotacao": [
                0.0,
                0.0,
                0.0
            ],
            "group_id": 1
        }
groups.append(newGroup)

newGroup = {
            "name": "fish",
            "deslocamento": [
                0.0,
                0.0,
                0.0
            ],
            "escala": [
                1.0,
                1.0,
                1.0
            ],
            "rotacao": [
                0.0,
                0.0,
                0.0
            ],
            "group_id": 2
        }
groups.append(newGroup)

In [36]:
global scene
glEnable(GL_DEPTH_TEST) ### importante para 3D

draw_mode = [GL_TRIANGLE_STRIP, GL_TRIANGLE_FAN]

scene = load_scene()

while not glfw.window_should_close(window):
   
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(1.0, 1.0, 1.0, 1.0)

    for obj in scene:

        if PolygonMode:
            glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
        else:
            glPolygonMode(GL_FRONT_AND_BACK,GL_FILL)

        if restrictMode:
            glfw.set_window_title(window, "Praia")
        else:
            glfw.set_window_title(window, f"name: {scene[selected_obj]['name']}, scene_id: {scene[selected_obj]['scene_id']}, uniform_scale: {uniform_scale}, eixo: {eixo}")

        if not obj['visible']:
            continue

        # Pegando valores do objeto
        tx, ty, tz = obj['deslocamento']
        ax, ay, az = obj['rotacao']
        sx, sy, sz = obj['escala']
        
        # Transformações
        # Translocação
        mat_t = mat_translocate(tx, ty, tz)
  
        # Rotação
        mat_rx = mat_rotation_x(ax)
        mat_ry = mat_rotation_y(ay)
        mat_rz = mat_rotation_z(az)
        
        mat_rot = multiplica_matriz3(mat_rx, mat_ry, mat_rz)

        # Escala
        mat_s = mat_scale(sx, sy, sz) 

        # Aplicação das Tranformações Locais
        mat_rot_scale = multiplica_matriz(mat_rot, mat_s)
        mat_transform = to_origin_and_back(scene, obj, mat_rot_scale, vertices, groups, False)
        mat_transform = multiplica_matriz(mat_transform, mat_t)

        # Operações sobre grupo
        gid = obj['group_id']
        if groupMode and gid != -1:
            # print(gid)

            gtx, gty, gtz = groups[gid]['deslocamento']
            gax, gay, gaz = groups[gid]['rotacao']
            gsx, gsy, gsz = groups[gid]['escala']
            
            # Transformações de Grupo
            # Translocação
            matg_t = mat_translocate(gtx, gty, gtz)
    
            # Rotação
            matg_rx = mat_rotation_x(gax)
            matg_ry = mat_rotation_y(gay)
            matg_rz = mat_rotation_z(gaz)
            
            matg_rot = multiplica_matriz3(matg_rx, matg_ry, matg_rz)

            # Escala
            matg_s = mat_scale(gsx, gsy, gsz) 
                
            # Aplica Transformações em Grupo
            mat_group_rot_scale = to_origin_and_back(scene, obj, multiplica_matriz(matg_rot, matg_s), vertices, groups, True)
            mat_transform = multiplica_matriz(mat_group_rot_scale, mat_transform)
            mat_transform = multiplica_matriz(matg_t, mat_transform)
        
        loc_transformation = glGetUniformLocation(program, "mat_transformation")
        glUniformMatrix4fv(loc_transformation, 1, GL_TRUE, mat_transform) 

        glUniform4f(loc_color, obj['cor'][0], obj['cor'][1], obj['cor'][2], obj['cor'][3])
        glDrawArrays(draw_mode[obj['draw_mode']], obj['vi'], obj['vt'])
    
    glfw.swap_buffers(window)
    glfw.poll_events()

glfw.terminate()

save_scene(scene)